In [5]:
# IMPORT REQUIRED MODULES 
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import scipy.stats 
from scipy.stats import skew, kurtosis
import numpy as np
import os

In [8]:
# Provide the filepaths for the ice discharge data provided by Mankoff et al. (2020) 
discharge_file = 'R:/KOGE_BUGT/github/data/mankoff_2020_ice_discharge/mankoff_data_gates/gate_D.csv'
error_file = 'R:/KOGE_BUGT/github/data/mankoff_2020_ice_discharge/mankoff_data_gates/gate_err.csv'
coverage_file = 'R:/KOGE_BUGT/github/data/mankoff_2020_ice_discharge/mankoff_data_gates/gate_coverage.csv'
output_dir = 'R:/KOGE_BUGT/github/data/mankoff_2020_ice_discharge/mankoff_data_gates/processed/'

# Provide the gate number for KBN, KBC, and KBS
gates = {"238": "KBN", "239": "KBC", "245": "KBS"}

# Define the start and end dates for the analysis
start_date = "2016-01-01"
end_date   = "2024-01-01"

# Loop through gates specified above and, for each gate, merge the discharge, error, and coverage data into a single dataframe
for gate, name in gates.items():
    discharge_df = pd.read_csv(discharge_file, usecols=['Date', gate]).rename(columns={gate: 'Discharge'})
    error_df = pd.read_csv(error_file, usecols=['Date', gate]).rename(columns={gate: 'Error'})
    coverage_df = pd.read_csv(coverage_file, usecols=['Date', gate]).rename(columns={gate: 'Coverage'})
    merged = (discharge_df.merge(error_df, on="Date").merge(coverage_df, on="Date"))
    merged["Date"] = pd.to_datetime(merged["Date"])
    filtered = merged[(merged["Coverage"] >= 0.5) & (merged["Date"] >= start_date) & (merged["Date"] < end_date)] # Filter for dates with >= 50% coverage
    output_file = os.path.join(output_dir, f"{name}_ice_discharge_gate{gate}.csv")
    filtered.to_csv(output_file, index=False)
    print(f"Saved {output_file}")

Saved R:/KOGE_BUGT/github/data/mankoff_2020_ice_discharge/mankoff_data_gates/processed/KBN_ice_discharge_gate238.csv
Saved R:/KOGE_BUGT/github/data/mankoff_2020_ice_discharge/mankoff_data_gates/processed/KBC_ice_discharge_gate239.csv
Saved R:/KOGE_BUGT/github/data/mankoff_2020_ice_discharge/mankoff_data_gates/processed/KBS_ice_discharge_gate245.csv


In [13]:
# Print summary statistics for each glacier
processed_dir = 'R:/KOGE_BUGT/github/data/mankoff_2020_ice_discharge/mankoff_data_gates/processed'
files = {"KBN": os.path.join(processed_dir, "KBN_ice_discharge_gate238.csv"),
    "KBC": os.path.join(processed_dir, "KBC_ice_discharge_gate239.csv"),
    "KBS": os.path.join(processed_dir, "KBS_ice_discharge_gate245.csv"),}
summary_stats = []
for name, path in files.items():
    df = pd.read_csv(path)
    df['Date'] = pd.to_datetime(df['Date'])
    date_max_discharge = df.loc[df['Discharge'].idxmax(), 'Date'].strftime('%Y-%m-%d')
    date_min_discharge = df.loc[df['Discharge'].idxmin(), 'Date'].strftime('%Y-%m-%d')
    stats = {'Glacier': name,
        'Mean Discharge': f"{df['Discharge'].mean():.2f}",
        'Median Discharge': f"{df['Discharge'].median():.2f}",
        'SD Discharge': f"{df['Discharge'].std():.2f}",
        'Min Discharge': f"{df['Discharge'].min():.2f}",
        'Date of Min Discharge': date_min_discharge,
        'Max Discharge': f"{df['Discharge'].max():.2f}",
        'Date of Max Discharge': date_max_discharge,
        'Skewness': f"{skew(df['Discharge'].dropna()):.2f}",
        'Kurtosis': f"{kurtosis(df['Discharge'].dropna()):.2f}"}
    summary_stats.append(stats)
summary_df = pd.DataFrame(summary_stats)
summary_df

,Glacier,Mean Discharge,Median Discharge,SD Discharge,Min Discharge,Date of Min Discharge,Max Discharge,Date of Max Discharge,Skewness,Kurtosis
0,KBN,4.17,4.16,0.13,3.87,2016-04-15,4.57,2021-08-17,0.46,0.05
1,KBC,16.05,16.38,1.04,13.77,2016-09-21,17.59,2021-08-17,-0.74,-0.76
2,KBS,7.32,7.31,0.14,6.92,2023-09-15,7.72,2020-02-24,0.37,-0.05
